# Full Course Review

从线性代数到 Transformer：用一张地图回顾整条知识链，再用一个"零框架"手写训练收尾——证明你已经掌握全部底层机制。


## 0. 环境配置与导入


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


## 1. 课程地图


| 模块 | 提供给后面的核心资产 | 在 Transformer 里的落点 |
|------|----------------------|------------------------|
| 线性代数 | 矩阵乘法、分解、特征值 | $QK^\mathsf{T}$、$W$ 参数矩阵、SVD/低秩 |
| 微积分 | 梯度、链式法则、Hessian | 反向传播、Adam、梯度裁剪 |
| 概率统计 | NLL、熵/KL、采样 | 交叉熵损失、softmax、温度采样 |
| 神经网络核心 | 训练循环、正则、初始化 | Block 的每个子层、优化器选择 |
| 架构 | 卷积/注意力/残差 | Transformer 本体 |

**一句话链路**：数据 →（概率）分布假设 →（线代）参数化变换 →（微积分）梯度学习 →（nn）训练工程 →（架构）组装成模型。


## 2. 核心概念速查表


| 概念 | 一句话 | 出处 |
|------|--------|------|
| 梯度下降 | 沿 $-\nabla L$ 走 | calculus 01/06 |
| 链式法则 | 梯度逐层连乘 | calculus 03/05 |
| 反向传播 | 从输出到输入的梯度流动 | calculus 05 |
| 负对数似然 | 所有损失函数的统一视角 | probability 05 |
| KL 散度 | 两个分布的额外编码代价 | probability 06 |
| 偏差-方差 | 复杂度与泛化的权衡 | nn-core 07 |
| 残差连接 | 恒等捷径让梯度穿越深层 | architectures 02 |
| 缩放注意力 | 除以 √d 保持方差稳定 | architectures 03 |
| 自回归 | 逐 token 条件概率连乘 | architectures 05 |


## 3. 收尾实验：零框架手写训练


不用 `torch.nn`、不用优化器、不用 autograd——只用 numpy 完成"前向 → 反向 → 更新"的完整循环。你在 nn-core 01 做过一次，现在应该能**独立**写出来。


In [ ]:
rng = np.random.default_rng(0)
n = 300
X0 = rng.standard_normal((n, 2)) + np.array([-2.0, 0.0])
X1 = rng.standard_normal((n, 2)) + np.array([2.0, 0.0])
X = np.vstack([X0, X1]); y = np.concatenate([np.zeros(n), np.ones(n)])

r = np.random.default_rng(1)
W1 = r.standard_normal((2, 8)) * 0.7; b1 = np.zeros(8)
W2 = r.standard_normal((8, 1)) * 0.7; b2 = np.zeros(1)

def sigmoid(z): return 1/(1+np.exp(-z))
lr = 0.5
losses = []
for epoch in range(1500):
    z1 = X @ W1 + b1; a1 = np.tanh(z1)
    z2 = a1 @ W2 + b2; a2 = sigmoid(z2)
    loss = -np.mean(y*np.log(a2.ravel()+1e-12) + (1-y)*np.log(1-a2.ravel()+1e-12))
    losses.append(loss)
    dz2 = (a2.ravel() - y).reshape(-1, 1) / n
    dW2 = a1.T @ dz2; db2 = dz2.sum(0)
    da1 = dz2 @ W2.T; dz1 = da1 * (1 - a1**2)
    dW1 = X.T @ dz1; db1 = dz1.sum(0)
    W1 -= lr*dW1; b1 -= lr*db1; W2 -= lr*dW2; b2 -= lr*db2

acc = ((a2.ravel() > 0.5) == y).mean()
print(f"零框架 MLP 最终 loss = {losses[-1]:.4f}，准确率 = {acc:.3f}")

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('epoch'); plt.ylabel('BCE loss')
plt.title('从零训练（无框架、无 autograd）：一切都在你手里')
plt.grid(alpha=0.3)


## 4. 下一步


本课程到此把"从数学到现代架构"的链路走完。可以继续的方向：

- **读源码**：读 PyTorch 的 `nn.TransformerEncoder`、`optim.AdamW` 实现，对照本课手写版
- **复现论文**：从"理解"到"复现"是质变——选一篇小模型论文（如 nanoGPT 思路）动手
- **深挖分支**：CV 的 ViT/扩散、NLP 的 MoE/RWKV、RL 的 PPO——它们都是本课组件的组合

**最后提醒**：遇到任何新概念，先问自己三个问题——它是什么分布/变换？它的梯度怎么算？它的代价函数是什么？


## 课后练习（毕业题）


1. **闭卷默写**：不查资料写出 Transformer Block 的前向公式（含归一化位置）。
2. **链路题**：解释"为什么交叉熵损失 + softmax + 反向传播"能训出语言模型（至少 5 句话，串起本课全部模块）。
3. **手推**：写出缩放注意力 $\mathrm{softmax}(QK^\mathsf{T}/\sqrt{d_k})V$ 对 $Q$ 的雅可比维度，并说明为什么需要除以 $\sqrt{d_k}$。
4. **工程题**：给你的 ToyGPT 加 batch 并行与梯度累积，说明各解决什么问题。
5. **教学题**：把"反向传播"用 3 句话讲给一个只学过微积分的同学，要求对方能听懂。
